In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches


Setup complete: 22 shared names ready in 50 s


In [2]:
# STANDIN B2
# The Part B black box with the settings chosen on validation in B2 (parts/10_baseline_models.ipynb).
# Dropped from the hand-in notebook, where the real B2 cell comes first.
from sklearn.ensemble import HistGradientBoostingClassifier

BOOST_SETTINGS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "random_state": SEED}
def make_boost():
    return HistGradientBoostingClassifier(**BOOST_SETTINGS)

boost = make_boost().fit(X_train, y_train)

In [3]:
# STANDIN D1
# The global SHAP ranking of step D1 (parts/30_explanation.ipynb): 1,000 random validation flows, seed 42.
# Dropped from the hand-in notebook, where the real D1 cell comes first.
import shap

GLOSSARY = pd.read_csv(TABLES / "T4_feature_glossary.csv", index_col="feature")
explainer = shap.TreeExplainer(boost)
sv = explainer(X_val.sample(1000, random_state=SEED))
shap_global = (pd.Series(np.abs(sv.values).mean(axis=0), index=FEATURES)
               .sort_values(ascending=False))

<!-- STEP F1 -->
## Part F: a knowledge-based layer (belief rule base)

A **belief rule base (BRB)** decides with a handful of readable rules instead of hundreds of trees:

> IF `feature 1` is *High* AND `feature 2` is *Low* THEN the risk is Low 10%, Medium 30%, High 60%.

Each input feature gets three **referential values**, Low, Medium and High. A flow's value is matched
against them (a value halfway between Low and Medium is half Low, half Medium), the rules fire according
to how well they match, and **Evidential Reasoning** combines their beliefs into one answer, including an
explicit **Unknown** part when the rules cannot decide. The brief asks for **two features with three
referential values each, so 3 × 3 = 9 rules**. The engine is Lab 3's tested `src/brbes.py`.

**Step F1: which two features?** The brief says: the two most important features from Part D. We go
down the global SHAP ranking (D1, computed on validation flows, so this choice never looks at the test
set) and keep the first two features that can carry three **distinct** referential values. These are
the training data's 5th percentile (Low), median (Medium) and 95th percentile (High). A feature is
skipped if it is a 0/1 flag, or if two of those values coincide and `brbes.make_levels` cannot separate
them (when many flows share one value, it first tries placing Medium and High among the values above
Low).

In [4]:
# STEP F1
if "src" not in sys.path:
    sys.path.insert(0, "src")
import brbes  # noqa: E402  the BRB engine from Lab 3 (src/brbes.py)

rows, TOP2 = [], []
for rank, feature in enumerate(shap_global.index, start=1):
    row = {"rank": rank, "feature": feature, "shap_share": shap_global[feature] / shap_global.sum(),
           "distinct_values": int(X_train[feature].nunique()),
           "percentiles_5_50_95": X_train[feature].quantile([0.05, 0.50, 0.95]).round(2).tolist()}
    if feature in FLAG_COLS:
        row.update(status="skipped", reason="0/1 flag: cannot have three levels")
    else:
        try:
            levels = brbes.make_levels(X_train[feature])
            row.update(status="chosen", reason="three distinct levels", levels=np.round(levels, 2).tolist())
            TOP2.append(feature)
        except ValueError:
            row.update(status="skipped", reason="two of the three levels coincide, even among the values above Low")
    rows.append(row)
    if len(TOP2) == 2:
        break

f1 = pd.DataFrame(rows).set_index("rank")
f1.to_csv(TABLES / "F1_feature_choice.csv")
print(f"TOP2 = {TOP2}")
f1

TOP2 = ['FWD Init Win Bytes', 'Bwd Packets/s']


,feature,shap_share,distinct_values,percentiles_5_50_95,status,reason,levels
rank,,,,,,,
1,Fwd Seg Size Min,0.420203,7,"[8.0, 32.0, 32.0]",skipped,"two of the three levels coincide, even among t...",NaN
2,FWD Init Win Bytes,0.214088,534,"[0.0, 5792.0, 26064.0]",chosen,three distinct levels,"[0.0, 5792.0, 26064.0]"
3,Bwd Packets/s,0.129336,132000,"[5.13, 296.67, 5633.8]",chosen,three distinct levels,"[5.13, 296.67, 5633.8]"


<!-- STEP F1 -->
### The two features for the BRB

| SHAP rank | Feature | Share of SHAP weight | Low / Medium / High (training 5th percentile / median / 95th percentile) |
|---:|---|---:|---|
| 1 | `Fwd Seg Size Min` | 42% | skipped: 8 / 32 / 32, only 7 distinct values, Medium and High coincide |
| 2 | **`FWD Init Win Bytes`** | 21% | **0 / 5,792 / 26,064** bytes |
| 3 | **`Bwd Packets/s`** | 13% | **5.1 / 296.7 / 5,633.8** packets per second |

`TOP2 = ["FWD Init Win Bytes", "Bwd Packets/s"]`.

**Why the most important feature is skipped.** `Fwd Seg Size Min` takes only 7 values, and 32 is both its
median and its 95th percentile (most benign flows have 32). Three distinct reference points cannot be
placed on it, not even among the values above its lowest value, so the next two features in the ranking
are used.

**What the two features mean.**
- **`FWD Init Win Bytes`**: the TCP window the initiating machine announces when it opens the connection,
  a setting of that machine's operating system (the second half of the attacker fingerprint, D1). In the
  E4 groups it is **Free**: the attacker can change it.
- **`Bwd Packets/s`**: how fast the other side sends packets back. It describes the victim's reaction
  (scans and floods get few or no replies) and is **Fixed**: the attacker cannot set it directly.

So the BRB will reason with one fingerprint feature and one behaviour feature.

**A limitation to watch in F2 and F3.** The attacker machines' window, 16,383 bytes, lies between Medium
(5,792) and High (26,064). The BRB reads it as about 48% Medium and 52% High, the same as any other value
near 16,000. Unlike the trees, which test for a narrow interval around 16,383 (B1), the BRB cannot single
out this exact value with these three reference points.

<!-- STEP F2 -->
## Part F: building the BRB

**The 9 rules.** Each feature has three levels, so there is one rule for every combination: IF
`FWD Init Win Bytes` is Low/Medium/High AND `Bwd Packets/s` is Low/Medium/High. In the lecture an expert
writes the rules' beliefs by hand; here they **come from the training data**, as in Lab 3:

1. Each training flow activates the rules according to how well its two values match the levels (a
   flow can be partly in two levels of each feature, so it can activate up to four rules).
2. For each rule, the **support** is how much training data activates it, and the **attack share** is
   the share of attacks among that data.
3. The attack share becomes beliefs over the risk levels Low / Medium / High: a share of 0 means fully
   Low, 0.5 fully Medium, 1 fully High, and values in between are split between the two nearest levels.
4. Rules with little support keep part of their belief as **Unknown**: the beliefs are scaled by
   support / (support + 20), so a rule backed by 20 flows keeps half of its belief Unknown.

**From beliefs to a decision.** For each flow the BRB's combined belief is turned into a risk score, the
**utility**: Low = 0, Medium = 50, High = 100, with the Unknown part counted halfway (the average of
treating it as Low and as High). A flow is called an attack if its utility is at least a **threshold**.
The threshold is chosen on the **validation** set: the value from 0 to 100 (steps of 0.5) with the best
macro-F1.

In [5]:
# STEP F2
levels = [brbes.make_levels(X_train[f]) for f in TOP2]
rb = brbes.build_rule_base(X_train[TOP2], y_train, levels)        # beliefs from the training data

rules = brbes.rules_table(rb)
rules.to_csv(TABLES / "F2_rules.csv", index=False, float_format="%.4f")
print(f"Levels: {TOP2[0]} {np.round(levels[0], 1).tolist()}, {TOP2[1]} {np.round(levels[1], 1).tolist()}")
rules[["rule", "if", "belief_Low", "belief_Medium", "belief_High", "ignorance", "support",
       "attack_share"]].style.format({"belief_Low": "{:.3f}", "belief_Medium": "{:.3f}",
                                      "belief_High": "{:.3f}", "ignorance": "{:.4f}",
                                      "support": "{:,.0f}", "attack_share": "{:.1%}"})

Levels: FWD Init Win Bytes [0.0, 5792.0, 26064.0], Bwd Packets/s [5.1, 296.7, 5633.8]


,rule,if,belief_Low,belief_Medium,belief_High,ignorance,support,attack_share
0,1,IF FWD Init Win Bytes is Low AND Bwd Packets/s is Low,0.029,0.967,0.000,0.0042,"4,686",48.5%
1,2,IF FWD Init Win Bytes is Low AND Bwd Packets/s is Medium,0.996,0.003,0.000,0.0013,"15,332",0.1%
2,3,IF FWD Init Win Bytes is Low AND Bwd Packets/s is High,0.996,0.000,0.000,0.0036,"5,521",0.0%
3,4,IF FWD Init Win Bytes is Medium AND Bwd Packets/s is Low,0.185,0.815,0.000,0.0004,"50,227",40.7%
4,5,IF FWD Init Win Bytes is Medium AND Bwd Packets/s is Medium,0.938,0.062,0.000,0.0004,"46,470",3.1%
5,6,IF FWD Init Win Bytes is Medium AND Bwd Packets/s is High,0.999,0.000,0.000,0.0010,"20,564",0.0%
6,7,IF FWD Init Win Bytes is High AND Bwd Packets/s is Low,0.000,0.227,0.772,0.0008,"25,260",88.6%
7,8,IF FWD Init Win Bytes is High AND Bwd Packets/s is Medium,0.053,0.941,0.000,0.0060,"3,309",47.3%
8,9,IF FWD Init Win Bytes is High AND Bwd Packets/s is High,0.998,0.000,0.000,0.0017,"11,693",0.0%


In [6]:
# STEP F2
# Threshold on the utility, chosen on the VALIDATION set (best macro-F1).
belief_val = brbes.infer(rb, X_val[TOP2])
u_val = brbes.utility(rb, belief_val)[2]                          # average utility, 0..100
BRB_THRESHOLD = brbes.choose_threshold(u_val, y_val)

grid = np.arange(0, 100.5, 0.5)
threshold_curve = pd.DataFrame([{"threshold": t, **score_proba(y_val, u_val / 100, t / 100)} for t in grid])
threshold_curve.to_csv(TABLES / "F2_threshold_val.csv", index=False, float_format="%.4f")

print(f"Chosen threshold: utility >= {BRB_THRESHOLD} means attack")
print("Validation scores at that threshold:",
      {k: round(v, 4) for k, v in score_proba(y_val, u_val / 100, BRB_THRESHOLD / 100).items()})
print(f"Unknown part on validation flows: mean {belief_val[:, 3].mean():.4f}, max {belief_val[:, 3].max():.4f}")
print(f"Distinct utility values on validation: {len(np.unique(u_val.round(6))):,}")

Chosen threshold: utility >= 48.5 means attack
Validation scores at that threshold: {'accuracy': 0.9631, 'macro_f1': 0.954, 'recall': 0.9839, 'pr_auc': 0.8946, 'far': 0.0443}
Unknown part on validation flows: mean 0.0008, max 0.0060
Distinct utility values on validation: 46,157


<!-- STEP F2 -->
### Reading the 9 rules

| Rule | `FWD Init Win Bytes` | `Bwd Packets/s` | Attack share in training | Belief Low / Medium / High | Support (flows) |
|---:|---|---|---:|---|---:|
| 1 | Low | Low | 49% | 0.03 / 0.97 / 0 | 4,687 |
| 2 | Low | Medium | 0.2% | 1.00 / 0 / 0 | 15,332 |
| 3 | Low | High | 0.02% | 1.00 / 0 / 0 | 5,521 |
| 4 | Medium | Low | 41% | 0.19 / 0.81 / 0 | 50,227 |
| 5 | Medium | Medium | 3.1% | 0.94 / 0.06 / 0 | 46,470 |
| 6 | Medium | High | 0.01% | 1.00 / 0 / 0 | 20,564 |
| 7 | **High** | **Low** | **89%** | 0 / 0.23 / **0.77** | 25,260 |
| 8 | High | Medium | 47% | 0.05 / 0.94 / 0 | 3,309 |
| 9 | High | High | 0.02% | 1.00 / 0 / 0 | 11,693 |

(The levels: `FWD Init Win Bytes` Low 0, Medium 5,792, High 26,064 bytes; `Bwd Packets/s` Low 5.1,
Medium 296.7, High 5,633.8 packets per second. Beliefs are rounded; the rest of each rule's belief is
Unknown, at most 0.006.)

**Do the rules make sense to a security person? Mostly yes.**
- **A victim that answers quickly means low risk** (rules 3, 6 and 9: almost no attacks, whatever the
  window). Normal services reply briskly. Scans, floods and many exploits get slow, few or no replies.
- **A slow reply rate combined with a large window is the one High-risk rule** (rule 7, 89% attacks).
  The attacker machines announce a window of 16,383 bytes, which falls between Medium and High, and their
  targets reply slowly.
- **A slow reply rate on its own is ambiguous** (rules 1, 4 and 8: 41 to 49% attacks, belief mostly
  Medium). Benign flows can also get slow replies, so the BRB stays undecided there, and says so.

**The decision.** The threshold chosen on validation is a utility of **48.5**, just below Medium (50).
Any flow whose belief is mostly Medium is therefore called an attack: the BRB leans towards catching
attacks rather than avoiding false alarms. On the validation set this gives **macro-F1 0.954, recall
0.984, PR-AUC 0.895 and FAR 0.044**, with two features and nine readable rules, against 0.971 / 0.992 /
0.981 / 0.028 for gradient boosting with 67 features and 100 trees.

**Unknown is tiny here** (mean 0.0008, at most 0.006 on validation flows): every rule is backed by at
least 3,309 training flows, so the data-driven beliefs leave almost nothing unexplained. Unknown becomes
large when an input is missing or a rule has little data behind it; step F4 shows the belief output for
one flow, and the optional extra X7 shows a flow with a missing input.